# ✏️ Handwritten Digit Recognizer
A deep learning model that reads handwritten digits (0–9) using the MNIST dataset and a Convolutional Neural Network (CNN).

**Run everything at once:** Runtime → Run all (`Ctrl+F9`)

| Step | What you'll do |
|------|----------------|
| 1 | Install & import libraries |
| 2 | Load the MNIST dataset |
| 3 | Explore & visualize the data |
| 4 | Preprocess |
| 5 | Build a CNN model |
| 6 | Train with early stopping |
| 7 | Evaluate & confusion matrix |
| 8 | Test on your own drawings (Gradio) |

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow gradio seaborn --quiet

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

print('✅ All libraries imported!')
print(f'   TensorFlow: {tf.__version__}')

## Step 2 — Load the MNIST Dataset

In [ ]:
# MNIST is built into Keras — no downloading needed!
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print('📊 Dataset loaded!')
print(f'   Training images : {X_train.shape}  → {len(X_train)} images of 28×28 pixels')
print(f'   Test images     : {X_test.shape}   → {len(X_test)} images of 28×28 pixels')
print(f'   Labels          : digits 0 through {y_test.max()}')

## Step 3 — Explore & Visualize the Data

In [ ]:
# Show 25 random training images
plt.figure(figsize=(10, 10))
for i in range(25):
    idx = np.random.randint(0, len(X_train))
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train[idx], cmap='gray')
    plt.title(f'Label: {y_train[idx]}', fontsize=10)
    plt.axis('off')
plt.suptitle('Sample MNIST Images', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Show class distribution
plt.figure(figsize=(8, 3))
unique, counts = np.unique(y_train, return_counts=True)
plt.bar(unique, counts, color='steelblue', edgecolor='white')
plt.title('Training samples per digit')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.xticks(range(10))
plt.tight_layout()
plt.show()

print('💡 MNIST is well-balanced — ~6,000 samples per digit.')

## Step 4 — Preprocess the Data

In [ ]:
# 1. Reshape: add channel dimension (CNNs expect height × width × channels)
#    28×28 → 28×28×1  (1 = grayscale)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# 2. Normalize: pixel values 0–255 → 0.0–1.0
#    Makes training faster and more stable
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# 3. One-hot encode labels
#    e.g. label 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat  = to_categorical(y_test,  num_classes=10)

print('✅ Preprocessing complete!')
print(f'   X_train shape : {X_train.shape}')
print(f'   y_train shape : {y_train_cat.shape}')
print(f'   Pixel range   : {X_train.min():.1f} → {X_train.max():.1f}')

## Step 5 — Build the CNN Model

In [ ]:
model = Sequential([
    # --- Block 1: detect simple edges and curves ---
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    # --- Block 2: detect more complex patterns ---
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    # --- Flatten & classify ---
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax')   # 10 output neurons, one per digit
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 6 — Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train_cat,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Training complete — stopped at epoch {len(history.history["loss"])}')

## Step 7 — Evaluate the Model

In [ ]:
# Test accuracy
loss, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'Test Loss     : {loss:.4f}\n')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred))

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 7b — Visualize Mistakes
> See which digits the model got wrong and why.

In [ ]:
# Find wrong predictions
wrong_idx = np.where(y_pred != y_test)[0]
print(f'Total mistakes: {len(wrong_idx)} out of {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)')

# Show 16 mistakes
plt.figure(figsize=(12, 8))
for i, idx in enumerate(wrong_idx[:16]):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f'True: {y_test[idx]}  Pred: {y_pred[idx]}', fontsize=9, color='red')
    plt.axis('off')
plt.suptitle('Mistakes the model made', fontsize=13)
plt.tight_layout()
plt.show()

## Step 8 — Save the Model

In [ ]:
model.save('digit_recognizer.keras')
print('✅ Model saved as digit_recognizer.keras')

# To reload:
# model = load_model('digit_recognizer.keras')

## Step 9 — Draw a Digit & Test It Live (Gradio)

In [ ]:
import gradio as gr

def predict_digit(img):
    if img is None:
        return 'Draw a digit first!'
    # img comes in as an RGBA numpy array from the sketchpad
    img = img[:, :, 0]                       # keep one channel
    img = img.astype('float32') / 255.0
    img = img.reshape(1, 28, 28, 1)
    probs = model.predict(img, verbose=0)[0]
    top3  = np.argsort(probs)[::-1][:3]
    result = ''
    for rank, digit in enumerate(top3):
        bar = '█' * int(probs[digit] * 20)
        result += f'#{rank+1}  Digit {digit}  {probs[digit]*100:5.1f}%  {bar}\n'
    return result

gr.Interface(
    fn=predict_digit,
    inputs=gr.Sketchpad(
        image_mode='RGBA',
        shape=(28, 28),
        label='Draw a digit (0–9)'
    ),
    outputs=gr.Textbox(label='Top 3 predictions', lines=4),
    title='Handwritten Digit Recognizer',
    description='Draw any digit from 0 to 9 and the model will predict it!'
).launch(debug=False)